# Original UTM raster vs. established compressors (no homemade codec)

Companion to `hpx_codec_bench.ipynb`, but on the **other side of the
resampling**: instead of compressing the HEALPix-tile representation, this
crops the ORIGINAL native-resolution UTM raster (Sentinel-2 B04, already
cached on disk as `data/<scene>_data.zarr` by `compress_test.ipynb`) down to
the exact geographic footprint of the same dominant HEALPix tile, and
compresses *that* -- so the two notebooks' RD results are directly
comparable: same ground footprint, HEALPix-resampled grid vs. native UTM
grid.

Two changes from `hpx_codec_bench.ipynb`:

- **No homemade codec.** `grad_dwt_zlib`/`grad_dwt_spiht` are dropped
  entirely -- conclusion after extensive tuning is that none of the
  in-house entropy coders (zlib, zero-tree, SPIHT) get anywhere close to
  established libraries even at high bitrate/near-lossless settings, so
  this benchmark only uses well-established, independently-maintained
  compressors (zfp, SZ3, JPEG 2000, JPEG XL, a plain DWT+zstd baseline).
- **Real geographic footprint, not a pixel-count heuristic.** The footprint
  crop uses the ACTUAL lon/lat of every fine-level (order 20) HEALPix cell
  in the dominant tile (`healpix_geo.nested.healpix_to_lonlat(...,
  ellipsoid="WGS84")` -- matching the ellipsoid the original UTM->HEALPix
  resampling already used by default), transformed into the scene's own UTM
  CRS, giving a real bounding box to crop the native raster to. This is
  more precise than the `ratio_20` same-pixel-count heuristic used
  elsewhere in this project.

Note `ellipsoid=True` is not a valid argument to `healpix_geo` -- it takes a
named ellipsoid string (`"WGS84"` or `"sphere"`); passing `True` raises a
`TypeError`. `"WGS84"` is used here to match
`healpix_resample.KNeighborsResampler`'s default, so the geometry is
self-consistent with how `data/all_image_cell.npy` was generated.

In [ ]:
%load_ext autoreload
%autoreload 2

# Minimal, independent deps -- this notebook does NOT import `hcc` at all
# (no homemade codec involved), so it doesn't need zarr>=3 or a kernel
# restart the way hpx_codec_bench.ipynb's first cell does.
try:
    import xarray as xr
except ImportError:
    !pip install xarray-eopf
    import xarray as xr
try:
    import pyproj
except ImportError:
    !pip install pyproj
    import pyproj
try:
    import healpix_geo
except ImportError:
    !pip install healpix-geo
    import healpix_geo
try:
    import pywt
except ImportError:
    !pip install PyWavelets
    import pywt
try:
    import zfpy
except ImportError:
    !pip install zfpy
    import zfpy
try:
    import pysz
except ImportError:
    !pip install pysz
    import pysz
try:
    import zstandard
except ImportError:
    !pip install zstandard


In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# --- colour-blind-safe palette (Okabe & Ito, 2008) + distinct markers -------
CODEC_STYLE = {
    "zfp":               dict(color="#0072B2", marker="o"),  # blue
    "sz3":                dict(color="#D55E00", marker="s"),  # vermillion
    "jpeg2000_lossy":     dict(color="#009E73", marker="^"),  # bluish green
    "jpeg2000_lossless":  dict(color="#000000", marker="*"),  # black
    "jpegxl_lossy":       dict(color="#CC79A7", marker="D"),  # reddish purple
    "jpegxl_lossless":    dict(color="#E69F00", marker="P"),  # orange
    "dwt_zstd":           dict(color="#56B4E9", marker="v"),  # sky blue
}
plt.rcParams["figure.figsize"] = (8, 5.5)
plt.rcParams["font.size"] = 11

In [ ]:
# --- crop the ORIGINAL UTM raster to the dominant HEALPix tile's real
# geographic footprint -------------------------------------------------------
# Reuses data/all_image_cell.npy (cell ids only -- we don't need the
# HEALPix-resampled values here) and the already-cached
# data/<scene>_data.zarr stores compress_test.ipynb writes. No network
# access, no re-derivation.
SCENE_NAMES = ["urban", "water", "forest", "agriculture", "snow_ice", "clouds", "ocean"]
LEVEL = 20  # must match the level used when all_image_cell.npy was built

cell_path = DATA_DIR / "all_image_cell.npy"
if not cell_path.exists():
    raise FileNotFoundError(
        f"Expected {cell_path} -- run compress_test.ipynb's cell 10 (data "
        "derivation) once first; this notebook only reuses its cached "
        "cell-id output, plus the data/<scene>_data.zarr stores it also "
        "writes (compress_test.ipynb cell 6)."
    )
cell_ids_all = np.load(cell_path)
assert cell_ids_all.shape[0] == len(SCENE_NAMES), (
    f"cell_ids_all.shape[0]={cell_ids_all.shape[0]} != len(SCENE_NAMES)="
    f"{len(SCENE_NAMES)} -- SCENE_NAMES must match the row order used when "
    "all_image_cell.npy was written."
)


def _src_crs(dt):
    """Same CRS lookup as compress_test.ipynb's add_latlon_to_dt (cell 5)."""
    try:
        crs_code = dt.attrs["other_metadata"]["horizontal_CRS_code"]
    except KeyError:
        crs_code = dt.attrs["other_metadata"]["horizontal_crs_code"]
    return pyproj.CRS.from_string(crs_code)


UTM_TILES: dict[str, np.ndarray] = {}
RAW_BYTES: dict[str, int] = {}  # per scene -- crop sizes differ, unlike the
                                 # fixed 1024x1024 HEALPix tiles

for k, name in enumerate(SCENE_NAMES):
    zarr_path = DATA_DIR / f"{name}_data.zarr"
    dt = xr.open_datatree(str(zarr_path), engine="zarr", consolidated=False, chunks={})

    cell_ids = cell_ids_all[k].astype(np.uint64)
    lon, lat = healpix_geo.nested.healpix_to_lonlat(cell_ids, LEVEL, ellipsoid="WGS84")

    src_crs = _src_crs(dt)
    to_utm = pyproj.Transformer.from_crs(pyproj.CRS.from_epsg(4326), src_crs, always_xy=True)
    utm_x, utm_y = to_utm.transform(lon, lat)

    b04 = dt["b04"]
    mask_x = (b04.x.values >= utm_x.min()) & (b04.x.values <= utm_x.max())
    mask_y = (b04.y.values >= utm_y.min()) & (b04.y.values <= utm_y.max())
    crop = b04.isel(x=np.where(mask_x)[0], y=np.where(mask_y)[0]).values.astype(np.float32)
    if np.isnan(crop).any():
        crop = np.where(np.isnan(crop), np.float32(np.nanmean(crop)), crop)

    UTM_TILES[name] = crop
    RAW_BYTES[name] = crop.nbytes
    print(f"{name:<12} footprint {mask_x.sum():4d}x{mask_y.sum():<4d} px "
          f"({crop.nbytes / 1e6:6.2f} MB)  "
          f"lon[{lon.min():.4f},{lon.max():.4f}] lat[{lat.min():.4f},{lat.max():.4f}]")

In [ ]:
# --- codec adapters ----------------------------------------------------------
from dataclasses import dataclass


@dataclass
class CodecRunResult:
    codec: str
    config: dict
    total_bytes: int
    encode_seconds: float
    decode_seconds: float
    reconstruction: np.ndarray


def _rmse(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[mask] - b[mask]) ** 2))) if mask.any() else np.nan


def encode_zfp(tile, *, tolerance):
    import zfpy
    tile = np.asarray(tile, dtype=np.float32)
    t0 = time.perf_counter()
    enc = zfpy.compress_numpy(tile, tolerance=float(tolerance))
    t1 = time.perf_counter()
    dec = zfpy.decompress_numpy(enc)
    t2 = time.perf_counter()
    return CodecRunResult("zfp", dict(tolerance=float(tolerance)), len(enc), t1 - t0, t2 - t1, dec)


def encode_sz3(tile, *, abs_error):
    from pysz import sz as _sz, szConfig, szErrorBoundMode
    tile = np.ascontiguousarray(np.asarray(tile, dtype=np.float32))
    config = szConfig()
    config.errorBoundMode = szErrorBoundMode.ABS
    config.absErrorBound = float(abs_error)
    t0 = time.perf_counter()
    compressed, _ = _sz.compress(tile, config)
    t1 = time.perf_counter()
    dec, _ = _sz.decompress(compressed, np.float32, tile.shape)
    t2 = time.perf_counter()
    total_bytes = int(getattr(compressed, "nbytes", None) or len(compressed))
    return CodecRunResult("sz3", dict(abs_error=float(abs_error)), total_bytes, t1 - t0, t2 - t1,
                           np.asarray(dec, dtype=np.float32))


def _to_u16(tile):
    tile = np.asarray(tile, dtype=np.float32)
    t_min, t_max = float(tile.min()), float(tile.max())
    span = (t_max - t_min) or 1.0
    return ((tile - t_min) / span * 65535).astype(np.uint16), t_min, span


def _from_u16(u16, t_min, span):
    return u16.astype(np.float32) / 65535.0 * span + t_min


def encode_jpeg2000_lossy(tile, *, quality):
    import io
    from PIL import Image
    u16, t_min, span = _to_u16(tile)
    factor = max(1, int(quality))  # never let it round to 0 (past bug, see main notebook)
    buf = io.BytesIO()
    t0 = time.perf_counter()
    Image.fromarray(u16).save(buf, format="jpeg2000", quality_mode="rates", quality_layers=[factor])
    t1 = time.perf_counter()
    total_bytes = buf.tell()
    buf.seek(0)
    dec_u16 = np.array(Image.open(buf), dtype=np.uint16)
    t2 = time.perf_counter()
    return CodecRunResult("jpeg2000_lossy", dict(quality=factor), total_bytes, t1 - t0, t2 - t1,
                           _from_u16(dec_u16, t_min, span))


def encode_jpeg2000_lossless(tile, **_ignored):
    import imagecodecs
    u16, t_min, span = _to_u16(tile)
    t0 = time.perf_counter()
    enc = imagecodecs.jpeg2k_encode(u16, level=0)
    t1 = time.perf_counter()
    dec_u16 = imagecodecs.jpeg2k_decode(enc)
    t2 = time.perf_counter()
    return CodecRunResult("jpeg2000_lossless", {}, len(enc), t1 - t0, t2 - t1,
                           _from_u16(dec_u16, t_min, span))


_jpegxl_warned = False


def encode_jpegxl_lossy(tile, *, level):
    # imagecodecs' `level` for jpegxl follows its shared 0-100 convention
    # across codecs (100 ~ visually lossless / near-lossless, not literally
    # lossless -- see encode_jpegxl_lossless for that). If the installed
    # imagecodecs version uses a different kwarg name, this fails loudly
    # once with a clear message instead of silently corrupting the sweep.
    global _jpegxl_warned
    import imagecodecs
    u16, t_min, span = _to_u16(tile)
    t0 = time.perf_counter()
    try:
        enc = imagecodecs.jpegxl_encode(u16, level=int(level))
    except TypeError as e:
        if not _jpegxl_warned:
            print(f"[jpegxl] encode signature mismatch ({e}) -- check your imagecodecs "
                  "version's jpegxl_encode kwargs and adjust this adapter. Skipping jpegxl.")
            _jpegxl_warned = True
        raise
    t1 = time.perf_counter()
    dec_u16 = imagecodecs.jpegxl_decode(enc)
    t2 = time.perf_counter()
    return CodecRunResult("jpegxl_lossy", dict(level=int(level)), len(enc), t1 - t0, t2 - t1,
                           _from_u16(dec_u16, t_min, span))


def encode_jpegxl_lossless(tile, **_ignored):
    import imagecodecs
    u16, t_min, span = _to_u16(tile)
    t0 = time.perf_counter()
    enc = imagecodecs.jpegxl_encode(u16, lossless=True)
    t1 = time.perf_counter()
    dec_u16 = imagecodecs.jpegxl_decode(enc)
    t2 = time.perf_counter()
    return CodecRunResult("jpegxl_lossless", {}, len(enc), t1 - t0, t2 - t1,
                           _from_u16(dec_u16, t_min, span))


def encode_dwt_zstd(tile, *, wavelet="db4", levels=4, quantize_bits):
    # Deliberately simple, well-known technique (PyWavelets + uniform
    # quantisation + zstd) -- a generic scale-correlation-exploiting
    # baseline, NOT the custom hcc SPIHT/zero-tree machinery this notebook
    # is moving away from.
    import pywt
    import zstandard as zstd
    tile = np.asarray(tile, dtype=np.float32)
    t0 = time.perf_counter()
    coeffs = pywt.wavedec2(tile, wavelet=wavelet, level=levels, mode="periodization")
    arr, slices = pywt.coeffs_to_array(coeffs)
    max_level = (1 << (quantize_bits - 1)) - 1
    scale = float(np.max(np.abs(arr))) or 1.0
    dtype_q = np.int8 if quantize_bits <= 8 else np.int16
    q = np.round(np.clip(arr / scale * max_level, -max_level, max_level)).astype(dtype_q)
    payload = zstd.ZstdCompressor(level=19).compress(q.tobytes())
    t1 = time.perf_counter()
    total_bytes = len(payload) + 4  # + float32 scale sidecar
    q2 = np.frombuffer(zstd.ZstdDecompressor().decompress(payload), dtype=dtype_q).reshape(arr.shape)
    arr2 = (q2.astype(np.float32) / max_level * scale).astype(np.float64)
    coeffs2 = pywt.array_to_coeffs(arr2, slices, output_format="wavedec2")
    dec = pywt.waverec2(coeffs2, wavelet=wavelet, mode="periodization").astype(np.float32)
    t2 = time.perf_counter()
    return CodecRunResult("dwt_zstd", dict(wavelet=wavelet, levels=levels, quantize_bits=quantize_bits),
                           total_bytes, t1 - t0, t2 - t1, dec)


def encode_uncompressed(tile, **_ignored):
    tile = np.asarray(tile, dtype=np.float32)
    return CodecRunResult("uncompressed", {}, tile.nbytes, 0.0, 0.0, tile.copy())

In [ ]:
# --- sweeps (kept modest -- this notebook is meant to iterate fast) --------
ZFP_SWEEP = [dict(tolerance=float(v)) for v in np.logspace(-5, -1, 8)]
SZ3_SWEEP = [dict(abs_error=float(v)) for v in np.logspace(-5, -1, 8)]
JPEG2000_LOSSY_SWEEP = [dict(quality=q) for q in (1, 2, 3, 5, 8, 12, 20, 35, 60, 100)]
JPEGXL_LOSSY_SWEEP = [dict(level=lv) for lv in (10, 25, 40, 55, 68, 78, 86, 92, 96, 99)]
DWT_ZSTD_SWEEP = [dict(quantize_bits=b) for b in (4, 6, 8, 10, 12, 14, 16)]

CODEC_RUNS = {
    "zfp": (encode_zfp, ZFP_SWEEP),
    "sz3": (encode_sz3, SZ3_SWEEP),
    "jpeg2000_lossy": (encode_jpeg2000_lossy, JPEG2000_LOSSY_SWEEP),
    "jpeg2000_lossless": (encode_jpeg2000_lossless, [dict()]),
    "jpegxl_lossy": (encode_jpegxl_lossy, JPEGXL_LOSSY_SWEEP),
    "jpegxl_lossless": (encode_jpegxl_lossless, [dict()]),
    "dwt_zstd": (encode_dwt_zstd, DWT_ZSTD_SWEEP),
    "uncompressed": (encode_uncompressed, [dict()]),
}

rows = []
for scene, tile in UTM_TILES.items():
    for codec, (fn, sweep) in CODEC_RUNS.items():
        for cfg in sweep:
            try:
                run = fn(tile, **cfg)
            except Exception as e:
                print(f"[skip] {codec} {cfg} on {scene}: {type(e).__name__}: {e}")
                continue
            rmse = _rmse(tile, run.reconstruction)
            rows.append(dict(
                scene=scene, codec=run.codec, config_json=json.dumps(run.config),
                total_bytes=run.total_bytes, rmse=rmse,
                encode_seconds=run.encode_seconds, decode_seconds=run.decode_seconds,
            ))
    print(f"done: {scene}  ({tile.shape[0]}x{tile.shape[1]})")

rd_df = pd.DataFrame(rows)
rd_df.to_csv(RESULTS_DIR / "utm_codec_bench.csv", index=False)
print(rd_df.shape, "rows saved to results/utm_codec_bench.csv")
rd_df.head()

In [ ]:
# --- Diagnostic: per-scene RD curves (size vs RMSE), one panel per scene --
# Unlike hpx_codec_bench.ipynb, crop sizes differ per scene here, so each
# panel normalises by ITS OWN scene's raw byte count (RAW_BYTES[scene]), not
# a single shared reference.
codec = {}
for k in rd_df.codec:
    codec[k] = 1
scene = {}
for k in rd_df.scene:
    scene[k] = 1
scene = [k for k in scene]
codec = [k for k in codec]

LOSSLESS_CODECS = {"jpeg2000_lossless", "jpegxl_lossless"}

n_cols = 3
n_rows = int(np.ceil(len(scene) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.3 * n_cols, 3.4 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, sc in zip(axes, scene):
    raw_bytes_sc = RAW_BYTES[sc]
    msize = 200
    for cd in codec:
        if cd == "uncompressed":
            continue
        g = rd_df[(rd_df.scene == sc) & (rd_df.codec == cd)]
        g = g.replace([np.inf, -np.inf], np.nan).dropna(subset=["rmse", "total_bytes"])
        g = g[g["rmse"] > 0]
        if g.empty:
            continue
        style = CODEC_STYLE.get(cd, dict(color="grey", marker="x"))
        if cd in LOSSLESS_CODECS or len(g) < 2:
            ax.scatter(g["rmse"], g["total_bytes"] / raw_bytes_sc, color=style["color"],
                       marker=style["marker"], s=msize, linewidth=0.7, zorder=5, label=cd)
            msize *= 0.5
        else:
            g = g.sort_values("rmse")
            ax.plot(g["rmse"], g["total_bytes"] / raw_bytes_sc, color=style["color"],
                    marker=style["marker"], markersize=5, linewidth=1.4, label=cd)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(sc, fontsize=10)
    ax.grid(True, which="both", linewidth=0.3, alpha=0.4)

for ax in axes[len(scene):]:
    ax.axis("off")

fig.supxlabel("RMSE (native UTM grid, log scale)")
fig.supylabel("Compressed size / this scene's raw crop size (log scale)")
handles, labels = [], []
for ax in axes[:len(scene)]:
    h, l = ax.get_legend_handles_labels()
    for hi, li in zip(h, l):
        if li not in labels:
            handles.append(hi)
            labels.append(li)
fig.legend(handles, labels, loc="lower center", ncol=min(len(labels), 4),
           bbox_to_anchor=(0.5, -0.04), fontsize=9, frameon=False)
fig.suptitle("Rate-distortion curves per scene -- native UTM crop", y=1.02)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIGURES_DIR / "fig_utm_rd_curves_per_scene.pdf", bbox_inches="tight")
fig.savefig(FIGURES_DIR / "fig_utm_rd_curves_per_scene.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Pareto frontier per (scene, codec) + log-log interpolation helpers ----
def pareto_frontier(df):
    clean = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["total_bytes", "rmse"])
    clean = clean.sort_values(["total_bytes", "rmse"])
    kept, best_rmse = [], np.inf
    for idx, row in clean.iterrows():
        if row["rmse"] < best_rmse:
            kept.append(idx)
            best_rmse = row["rmse"]
    return clean.loc[kept].sort_values("rmse")


FRONTIERS = {
    (scene, codec): pareto_frontier(g)
    for (scene, codec), g in rd_df.groupby(["scene", "codec"])
}


def interp_size_at_rmse(frontier, target_rmse):
    f = frontier[["rmse", "total_bytes"]].dropna().sort_values("rmse").drop_duplicates("rmse")
    if len(f) < 2 or not (f["rmse"].min() <= target_rmse <= f["rmse"].max()):
        return np.nan
    return float(np.exp(np.interp(np.log(target_rmse), np.log(f["rmse"]), np.log(f["total_bytes"]))))


def interp_rmse_at_size(frontier, target_bytes):
    f = frontier[["total_bytes", "rmse"]].dropna().sort_values("total_bytes").drop_duplicates("total_bytes")
    if len(f) < 2 or not (f["total_bytes"].min() <= target_bytes <= f["total_bytes"].max()):
        return np.nan
    return float(np.exp(np.interp(np.log(target_bytes), np.log(f["total_bytes"]), np.log(f["rmse"]))))


COMPRESSOR_CODECS = [c for c in CODEC_RUNS if c != "uncompressed"]

In [ ]:
# --- Figure "storage at constant RMSE": pick target RMSEs from a robust
# common range (15th-85th percentile of ALL observed RMSE, see
# hpx_codec_bench.ipynb for why a strict per-scene/codec intersection is too
# fragile), then bar-chart size AS A FRACTION OF EACH SCENE'S OWN raw crop
# size (crop sizes differ per scene here, unlike the fixed HEALPix tiles) --
_all_rmse = rd_df.replace([np.inf, -np.inf], np.nan)["rmse"].dropna()
_all_rmse = _all_rmse[_all_rmse > 0]
_lo, _hi = np.percentile(_all_rmse, [15, 85])
if _lo < _hi:
    TARGET_RMSE_VALUES = list(np.geomspace(_lo, _hi, 4))
else:
    TARGET_RMSE_VALUES = []
    print("[warn] degenerate RMSE range -- Figure B will be empty.")
print("Target RMSE values:", TARGET_RMSE_VALUES)

matched_rmse_rows = []
for target in TARGET_RMSE_VALUES:
    for codec in COMPRESSOR_CODECS:
        fracs = []
        for s in UTM_TILES:
            if (s, codec) not in FRONTIERS:
                continue
            b = interp_size_at_rmse(FRONTIERS[(s, codec)], target)
            if np.isfinite(b):
                fracs.append(b / RAW_BYTES[s])
        if not fracs:
            continue
        matched_rmse_rows.append(dict(
            target_rmse=target, codec=codec,
            median_frac=float(np.median(fracs)),
            lo_frac=float(np.min(fracs)), hi_frac=float(np.max(fracs)),
            n_scenes=len(fracs),
        ))
matched_rmse_df = pd.DataFrame(matched_rmse_rows)
matched_rmse_df.to_csv(RESULTS_DIR / "utm_matched_rmse.csv", index=False)
matched_rmse_df

In [ ]:
fig, ax = plt.subplots()
if not matched_rmse_df.empty:
    codecs_present = [c for c in COMPRESSOR_CODECS if c in matched_rmse_df.codec.unique()]
    width = 0.8 / max(1, len(codecs_present))
    x0 = np.arange(len(TARGET_RMSE_VALUES))
    for i, codec in enumerate(codecs_present):
        sub = matched_rmse_df[matched_rmse_df.codec == codec].set_index("target_rmse")
        sub = sub.reindex(TARGET_RMSE_VALUES)
        ax.bar(x0 + i * width, sub["median_frac"], width=width, label=codec,
               color=CODEC_STYLE[codec]["color"])
    ax.axhline(1.0, color="grey", linestyle=":", linewidth=1, label="uncompressed")
    ax.set_xticks(x0 + width * (len(codecs_present) - 1) / 2)
    ax.set_xticklabels([f"{t:.2g}" for t in TARGET_RMSE_VALUES])
    ax.set_xlabel("Target RMSE (native UTM grid)")
    ax.set_ylabel("Compressed size / raw crop size (median across scenes)")
    ax.set_yscale("log")
    ax.legend(fontsize=8, ncol=2)
    ax.set_title("Storage at constant RMSE -- native UTM crop")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig_storage_at_constant_rmse.pdf")
    fig.savefig(FIGURES_DIR / "fig_storage_at_constant_rmse.png", dpi=150)
plt.show()

In [ ]:
# --- Figure "error at constant size": fixed size budgets, as a fraction of
# EACH SCENE'S OWN raw crop size ---------------------------------------------
FILE_BUDGET_FRACTIONS = [0.5, 0.25, 0.1, 0.05, 0.02]

matched_size_rows = []
for frac in FILE_BUDGET_FRACTIONS:
    for codec in COMPRESSOR_CODECS:
        rmses = [interp_rmse_at_size(FRONTIERS[(s, codec)], frac * RAW_BYTES[s])
                 for s in UTM_TILES if (s, codec) in FRONTIERS]
        rmses = [r for r in rmses if np.isfinite(r)]
        if not rmses:
            continue
        matched_size_rows.append(dict(
            budget_fraction=frac, codec=codec,
            median_rmse=float(np.median(rmses)),
            lo_rmse=float(np.min(rmses)), hi_rmse=float(np.max(rmses)),
            n_scenes=len(rmses),
        ))
matched_size_df = pd.DataFrame(matched_size_rows)
matched_size_df.to_csv(RESULTS_DIR / "utm_matched_size.csv", index=False)

fig, ax = plt.subplots()
for codec in COMPRESSOR_CODECS:
    sub = matched_size_df[matched_size_df.codec == codec].sort_values("budget_fraction")
    if sub.empty:
        continue
    style = CODEC_STYLE[codec]
    ax.plot(sub["budget_fraction"] * 100, sub["median_rmse"],
             color=style["color"], marker=style["marker"], label=codec)
    ax.fill_between(sub["budget_fraction"] * 100, sub["lo_rmse"], sub["hi_rmse"],
                     color=style["color"], alpha=0.15)
ax.set_xlabel("File budget (% of each scene's raw crop size)")
ax.set_ylabel("RMSE (native UTM grid)")
ax.set_yscale("log")
ax.legend(fontsize=8, ncol=2)
ax.set_title("Error at constant file size -- native UTM crop")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_error_at_constant_size.pdf")
fig.savefig(FIGURES_DIR / "fig_error_at_constant_size.png", dpi=150)
plt.show()
matched_size_df